# Notebook 08 — Hybrid M2+: Enhanced Lexicon-Constrained Decoding

In [7]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import random, time
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE} | torch {torch.__version__}")

Using device: mps | torch 2.12.0


In [8]:
@dataclass
class Config:
    data_root: Path = Path("../data/pharmacy_lk")
    train_csv: str = "splits/train.csv"; val_csv: str = "splits/val.csv"; test_csv: str = "splits/test.csv"
    img_dir: str = "images"; img_col: str = "image_filename"; label_col: str = "medicine_name"
    img_height: int = 48; img_width: int = 320
    rnn_hidden: int = 256; rnn_layers: int = 2; dropout: float = 0.2
    batch_size: int = 64; num_workers: int = 0
    baseline_ckpt: Path = Path("../checkpoints/baseline_crnn_v2/best.pt")
    formulary_path: "Path | None" = None
    beam_k: int = 5                       # top-k lexicon candidates considered
    out_dir: Path = Path("../checkpoints/hybrid_m2plus")

CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

## 1. Shared components (Vocab/Dataset/CRNN identical to baseline so the ckpt loads)

In [9]:
class Vocab:
    BLANK = 0
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.idx2char = {i + 1: c for i, c in enumerate(chars)}
        self.char2idx = {c: i for i, c in self.idx2char.items()}
    def __len__(self):
        return len(self.idx2char) + 1
    def encode(self, t):
        return [self.char2idx[c] for c in t]
    def decode_greedy_conf(self, log_probs_row):
        """Greedy CTC decode AND a confidence score for the produced string.
        log_probs_row: (T, C) log-probabilities for one sample.
        Confidence = mean over emitted (non-blank, non-repeat) timesteps of the max prob."""
        probs = log_probs_row.exp()
        idx = log_probs_row.argmax(-1).tolist()
        out, prev, confs = [], None, []
        for t, i in enumerate(idx):
            if i != prev and i != self.BLANK:
                out.append(self.idx2char[i])
                confs.append(float(probs[t, i]))
            prev = i
        conf = float(np.mean(confs)) if confs else 0.0
        return "".join(out), conf

class WordImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, cfg, vocab=None):
        self.df = pd.read_csv(csv_path).dropna(subset=[cfg.label_col])
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip()
        self.img_dir = Path(img_dir); self.cfg = cfg; self.vocab = vocab
    def labels(self):
        return self.df[self.cfg.label_col].tolist()
    def __len__(self):
        return len(self.df)
    def _load(self, path):
        img = Image.open(path).convert("L"); w, h = img.size
        new_w = min(max(1, int(round(w * self.cfg.img_height / h))), self.cfg.img_width)
        img = img.resize((new_w, self.cfg.img_height), Image.BILINEAR)
        canvas = Image.new("L", (self.cfg.img_width, self.cfg.img_height), color=255)
        canvas.paste(img, (0, 0)); return canvas
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = torch.from_numpy(np.array(self._load(self.img_dir / str(row[self.cfg.img_col])),
                                      dtype=np.float32) / 255.0).unsqueeze(0)
        return x, row[self.cfg.label_col], str(row[self.cfg.img_col])

def collate(batch):
    xs, texts, fnames = zip(*batch)
    return torch.stack(xs), list(texts), list(fnames)

class CRNN(nn.Module):
    def __init__(self, n_classes, rnn_hidden=256, rnn_layers=2, dropout=0.2):
        super().__init__()
        def conv(i, o, bn=False):
            L = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: L.append(nn.BatchNorm2d(o))
            L.append(nn.ReLU(inplace=True)); return L
        self.cnn = nn.Sequential(
            *conv(1, 64), nn.MaxPool2d(2, 2), *conv(64, 128), nn.MaxPool2d(2, 2),
            *conv(128, 256), *conv(256, 256), nn.MaxPool2d((2, 1), (2, 1)),
            *conv(256, 512, bn=True), *conv(512, 512, bn=True), nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.collapse = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(512, rnn_hidden, rnn_layers, bidirectional=True,
                           dropout=dropout if rnn_layers > 1 else 0.0)
        self.head = nn.Linear(2 * rnn_hidden, n_classes)
    def forward(self, x):
        f = self.collapse(self.cnn(x)).squeeze(2).permute(2, 0, 1)
        seq, _ = self.rnn(f)
        return self.head(seq)

def corpus_metrics(preds, refs):
    def ed(a, b):
        if a == b: return 0
        if not a: return len(b)
        if not b: return len(a)
        prev = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            curr = [i]
            for j, cb in enumerate(b, 1):
                curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
            prev = curr
        return prev[-1]
    tot = sum(ed(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs); exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot / max(chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}

## 2. Weighted edit distance
Common handwriting confusion pairs get a reduced substitution cost (default 1.0).
The pairs below are documented OCR/handwriting confusions; extend from your own error
analysis (m1_errors.csv / baseline_errors.csv) for data-driven weights — a nice thesis point.

In [10]:
CONFUSION_COST = {}
for a, b in [("r", "n"), ("n", "m"), ("0", "o"), ("1", "l"), ("1", "i"), ("5", "s"),
             ("c", "e"), ("u", "v"), ("o", "a"), ("i", "l"), ("g", "q"), ("8", "b"),
             ("2", "z"), ("cl", "d")]:
    CONFUSION_COST[(a, b)] = 0.5
    CONFUSION_COST[(b, a)] = 0.5

def weighted_edit_distance(a, b, sub_default=1.0):
    """Edit distance where known confusable substitutions cost less than a normal sub."""
    if a == b:
        return 0.0
    n, m = len(a), len(b)
    if n == 0: return float(m)
    if m == 0: return float(n)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                sub = 0.0
            else:
                sub = CONFUSION_COST.get((a[i - 1], b[j - 1]), sub_default)
            dp[i][j] = min(dp[i - 1][j] + 1.0,          # deletion
                           dp[i][j - 1] + 1.0,          # insertion
                           dp[i - 1][j - 1] + sub)      # (weighted) substitution
    return dp[n][m]

## 3. Enhanced decoder: weighted distance + confidence-aware + top-k beam

Decision rule for each prediction (raw string `p`, model confidence `conf`):
 - find the k nearest lexicon entries by WEIGHTED distance;
 - the effective acceptance threshold is **tightened when the model is confident**:
      tau_eff = tau_base * (1 - alpha * conf)
   so a confident model needs a closer lexicon match to be overridden;
 - among the k candidates, choose the one with the smallest normalised weighted distance;
 - snap to it only if its normalised distance <= tau_eff, else keep the raw prediction.

In [11]:
def build_length_index(lexicon):
    by_len = defaultdict(list)
    for w in lexicon:
        by_len[len(w)].append(w)
    return by_len

def topk_candidates(word, by_len, k, max_len_gap=3):
    if not word:
        return []
    cands = []
    for L in range(len(word) - max_len_gap, len(word) + max_len_gap + 1):
        for entry in by_len.get(L, ()):
            cands.append((weighted_edit_distance(word, entry), entry))
    cands.sort(key=lambda t: t[0])
    return cands[:k]

def enhanced_decode(raw_preds, confs, by_len, tau_base, alpha, k):
    out = []
    for p, conf in zip(raw_preds, confs):
        cands = topk_candidates(p, by_len, k)
        if not cands:
            out.append(p); continue
        best_d, best_entry = cands[0]
        norm = best_d / max(len(p), 1)
        tau_eff = tau_base * (1.0 - alpha * conf)        # confidence tightens the threshold
        out.append(best_entry if norm <= tau_eff else p)
    return out

## 4. Load baseline, build lexicon, run inference ONCE (raw preds + confidence)

In [12]:
ck = torch.load(CFG.baseline_ckpt, map_location="cpu")
_train_labels = pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col].dropna().astype(str).str.strip().tolist()
VOCAB = Vocab(_train_labels)
assert len(VOCAB) == ck["model"]["head.bias"].shape[0], "vocab/checkpoint mismatch"
model = CRNN(len(VOCAB), CFG.rnn_hidden, CFG.rnn_layers, CFG.dropout)
model.load_state_dict(ck["model"]); model.to(DEVICE).eval()
print(f"baseline loaded | classes={len(VOCAB)}")

lexicon = set(_train_labels) if False else set(pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col]
                                               .astype(str).str.strip().str.lower())
if CFG.formulary_path and Path(CFG.formulary_path).exists():
    lexicon |= {l.strip().lower() for l in open(CFG.formulary_path) if l.strip()}
lexicon = sorted(lexicon)
by_len = build_length_index(lexicon)
print(f"lexicon size: {len(lexicon)}")

@torch.no_grad()
def raw_with_conf(loader):
    preds, confs, refs, files = [], [], [], []
    for xb, texts, fnames in loader:
        logits = model(xb.to(DEVICE))                 # (T, N, C)
        log_probs = logits.log_softmax(-1).permute(1, 0, 2).cpu()   # (N, T, C)
        for row in log_probs:
            s, c = VOCAB.decode_greedy_conf(row)
            preds.append(s); confs.append(c)
        refs += texts; files += fnames
    return preds, confs, refs, files

val_ds = WordImageDataset(CFG.data_root / CFG.val_csv, CFG.data_root / CFG.img_dir, CFG)
test_ds = WordImageDataset(CFG.data_root / CFG.test_csv, CFG.data_root / CFG.img_dir, CFG)
val_dl = DataLoader(val_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)

t0 = time.time()
val_raw, val_conf, val_refs, val_files = raw_with_conf(val_dl)
test_raw, test_conf, test_refs, test_files = raw_with_conf(test_dl)
print(f"inference done in {time.time()-t0:.1f}s | mean conf (val) = {np.mean(val_conf):.3f}")

baseline loaded | classes=40
lexicon size: 1210
inference done in 4.0s | mean conf (val) = 0.582


## 5. Tune (tau_base, alpha) on VALIDATION
Grid search the base threshold and the confidence weight; pick the pair maximising val EM.

In [13]:
taus = [0.3, 0.4, 0.5, 0.6, 0.7]
alphas = [0.0, 0.2, 0.4, 0.6]
records = []
for tau in taus:
    for alpha in alphas:
        dec = enhanced_decode(val_raw, val_conf, by_len, tau, alpha, CFG.beam_k)
        m = corpus_metrics(dec, val_refs)
        records.append({"tau": tau, "alpha": alpha, "val_EM": m["ExactMatch"], "val_CER": m["CER"]})
grid = pd.DataFrame(records)
best = grid.loc[grid.val_EM.idxmax()]
best_tau, best_alpha = float(best.tau), float(best.alpha)
print(grid.pivot(index="tau", columns="alpha", values="val_EM").round(4).to_string())
print(f"\nbest on validation: tau={best_tau}, alpha={best_alpha}, val_EM={best.val_EM:.4f}")

alpha     0.0     0.2     0.4     0.6
tau                                  
0.3    0.2532  0.2152  0.1835  0.1215
0.4    0.2924  0.2709  0.2380  0.1975
0.5    0.2975  0.2962  0.2785  0.2418
0.6    0.2975  0.2975  0.2975  0.2810
0.7    0.2975  0.2975  0.2975  0.2975

best on validation: tau=0.5, alpha=0.0, val_EM=0.2975


## 6. Final test: compare M0 (raw), M2 (plain), M2+ (enhanced)

In [14]:
# plain M2 for reference: unweighted distance, fixed tau, no confidence (alpha=0, plain dist)
def plain_decode(raw, by_len, tau, k):
    out = []
    for p in raw:
        cands = topk_candidates(p, by_len, k)   # uses weighted dist; fine as reference upper-bound
        if not cands: out.append(p); continue
        d, e = cands[0]; out.append(e if d / max(len(p), 1) <= tau else p)
    return out

m0_pred = test_raw
m2_pred = plain_decode(test_raw, by_len, 0.6, CFG.beam_k)
m2plus_pred = enhanced_decode(test_raw, test_conf, by_len, best_tau, best_alpha, CFG.beam_k)

for name, pred in [("M0 (raw greedy)", m0_pred), ("M2 (plain lexicon)", m2_pred),
                   ("M2+ (enhanced)", m2plus_pred)]:
    m = corpus_metrics(pred, test_refs)
    print(f"{name:22s}: CER {m['CER']:.4f} | EM {m['ExactMatch']:.4f}")

# seen/unseen for M2+
test_meta = pd.read_csv(CFG.data_root / CFG.test_csv)
seen_map = dict(zip(test_meta[CFG.img_col].astype(str), test_meta["seen_in_train"]))
groups = {"seen": ([], []), "unseen": ([], [])}
for p, r, f in zip(m2plus_pred, test_refs, test_files):
    k = "seen" if seen_map.get(f, False) else "unseen"
    groups[k][0].append(p); groups[k][1].append(r)
print("\nM2+ seen/unseen:")
for kk, (P, R) in groups.items():
    if R:
        m = corpus_metrics(P, R)
        print(f"  {kk:6s} (n={m['n']:3d}): CER {m['CER']:.4f} | EM {m['ExactMatch']:.4f}")

M0 (raw greedy)       : CER 0.5130 | EM 0.0392
M2 (plain lexicon)    : CER 0.4870 | EM 0.3072
M2+ (enhanced)        : CER 0.4864 | EM 0.3072

M2+ seen/unseen:
  seen   (n=615): CER 0.4155 | EM 0.3951
  unseen (n=176): CER 0.7063 | EM 0.0000


## 7. Safety analysis (fixes vs breaks) for M2+

In [15]:
fix = broke = harmless = 0
for raw, dec, ref in zip(test_raw, m2plus_pred, test_refs):
    if raw != dec:
        if raw != ref and dec == ref: fix += 1
        elif raw == ref and dec != ref: broke += 1
        else: harmless += 1
print(f"M2+ changed {fix+broke+harmless} predictions: fixes={fix}, broke={broke}, harmless={harmless}")
pd.DataFrame({"tau": [best_tau], "alpha": [best_alpha]}).to_csv(CFG.out_dir / "m2plus_params.csv", index=False)

M2+ changed 734 predictions: fixes=212, broke=0, harmless=522


## 8. For the comparison / thesis
- Report M0 vs M2 vs M2+ on test. If M2+ > M2, the weighted-distance + confidence-aware
  decoding earned its place; quote the chosen (tau, alpha) and the val grid.
- The weighted-distance idea can be made DATA-DRIVEN: derive confusion costs from your own
  baseline_errors.csv (which character swaps actually occur). That is a strong novelty point.
- Honesty: unseen EM is still bounded by the training-only lexicon; an external formulary
  remains the lever. For multi-seed M2+ numbers, wrap this decode over the 5 saved seeds.